# 07 | Construcción de los modelos

> **Cuaderno de reproducción (posterior).** Generado el 2026-10-23 como parte de la entrega docente. El repositorio original no contenía cuadernos versionados; estos reconstruyen los análisis a partir de los scripts y módulos existentes.

**Tipo de ejecución: CONSULTA — lee resultados previamente generados; NO entrena modelos.**

In [1]:
import os
from pathlib import Path

def _raiz_repositorio() -> Path:
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / 'data' / 'raw' / 'interseccion_terrenos_zhg_r1_yacopi.csv').exists():
            return p
    raise FileNotFoundError('No se encontro la raiz del repositorio (busca data/raw)')

REPO = _raiz_repositorio()
print('Raiz del repositorio resuelta (carpeta):', REPO.name, '| data/raw encontrado')

Raiz del repositorio resuelta (carpeta): yacopi-project | data/raw encontrado


## Código real del protocolo v2 (actual)

`ml/src/yacopi_ml/valor/modelos.py` define `preprocesador`, `entrenar_modelos` (Dummy, Ridge log, Árbol, RandomForest, CatBoost) y `predecir`. Es el módulo ejecutado por train.py y audit_r2.py.

In [2]:
src = (REPO / 'ml/src/yacopi_ml/valor/modelos.py').read_text(encoding='utf-8')
print(src)

"""Modelos compartidos del experimento VALOR_TERRENO_Ha (experimental).

Centraliza construccion de modelos, preprocesador y metricas para que
train.py y audit_r2.py usen exactamente el mismo protocolo.
"""

from __future__ import annotations

import numpy as np
import pandas as pd
from catboost import CatBoostRegressor
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeRegressor

from yacopi_ml.valor.dataset import CATEGORICAS, NUMERICAS

SEED = 20261023


def metricas(y_true, y_pred) -> dict:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    ape = np.abs((y_true -

## Código histórico de Fase 2 (NO es el protocolo actual)

`ml/src/yacopi_ml/models.py` pertenece al experimento proxy histórico con búsqueda de hiperparámetros (seed 42). Se conserva como evidencia, no se usa en el protocolo v2 (que usa hiperparámetros fijos y semilla 20261023).

In [3]:
src = (REPO / 'ml/src/yacopi_ml/models.py').read_text(encoding='utf-8')
print(src[:2600])

"""Definición de los modelos candidatos y sus distribuciones de tuning."""

from catboost import CatBoostRegressor
from scipy.stats import randint, uniform
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeRegressor

from .preprocess import preprocess_sklearn
from .schema import CATEGORICAL_FEATURES

SEED = 42


class CatBoostWrapper(BaseEstimator, RegressorMixin):
    """Envoltorio sklearn-compatible de CatBoost (cat_features clonable)."""

    def __init__(self, cat_features, iterations=500, depth=6,
                 learning_rate=0.07, l2_leaf_reg=3.0, random_seed=SEED):
        self.cat_features = cat_features
        self.iterations = iterations
        self.depth = depth
        self.learning_rate = learning_rate
        self.l2_leaf_reg = l2_leaf_reg
        self.random_s